# Formalia

Please read the [assignment overview page](https://laura.alessandretti.com/comsocsci2026/wiki_pages/Assignments.html) carefully before proceeding. The page contains information about formatting (including formats etc), group sizes, and many other aspects of handing in the assignment. 

__If you fail to follow these simple instructions, it will negatively impact your grade!__

**Due date and time**: The assignment is due on Apr 7th at 23:59. Hand in your Jupyter notebook file (with extension `.ipynb`) via DTU Learn _(Assignment 2)_. 

Remember to include in the first cell of your notebook:
* the link to your group's Git repository 
* group members' contributions

# Part 1: Mixing Patterns and Assortativity

> __Exercise 1: Mixing Patterns and Assortativity__  
>
> __Part 1: Assortativity Coefficient__ 
> 1. *Calculate the Assortativity Coefficient* for the network based on the country of each node. Implement the calculation using the formula provided during the lecture, also available in [this paper](https://arxiv.org/pdf/cond-mat/0209450.pdf) (equation 2, here for directed networks). **Do not use the NetworkX implementation.**
>

Firstly, we need to make a network. We use the provided dataset to make avoid running into unnecessay issues.


In [5]:
import pandas as pd
import ast
from itertools import combinations
from collections import defaultdict

# load data
CSS_authors=pd.read_csv('data/final_authors.csv')
CSS_papers=pd.read_csv('data/final_papers.csv')
pair_counts=defaultdict(int)

In [11]:
# Compute all pairs of authors, who have published a paper together and count how many times they have
i=0
for authors in CSS_papers['author_ids'].apply(ast.literal_eval):
    for pair in combinations(authors,2):
        if None in pair:
            continue
        pair=tuple(sorted(pair))
        pair_counts[pair]+=1
    i+=1


In [16]:
our_authors=list(CSS_authors['id'].astype(str))

ohno=0

weighted_list=[]
for authors,number in pair_counts.items():
    author1,author2=authors
    # check if authors actually in our authors dataset
    if author1 not in our_authors: 
        ohno+=1
        continue
    if author2 not in our_authors: 
        ohno+=1
        continue
    weighted_list.append((author1,author2,number))

print(ohno)
print(len(weighted_list))

990184
51565


Now we create the graph, edges and country code data

In [27]:
import networkx as nx
G=nx.Graph()
G.add_weighted_edges_from(weighted_list)

import json
# set node attributes
for i,author_id in enumerate(G.nodes):        
    G.nodes[author_id]['country_code']=CSS_authors['country_code'][i]



# save network
JSON_data=nx.node_link_data(G)
with open('graph.json','w') as file:
    json.dump(JSON_data,file,indent=2)


Now we can calculate the assortativity degree

In [31]:
e=defaultdict(int)
a=defaultdict(int)
# get all e and a values for each country code
for edge in G.edges(data=True):
    n1=edge[0]
    n2=edge[1]
    country1=G.nodes[n1]['country_code']
    country2=G.nodes[n2]['country_code']
    if country1==country2:
        e[country1]+=1
    a[country1]+=1
    a[country2]+=1

total_edges = G.number_of_edges()
stubs=2*total_edges
sum_e=sum(v/total_edges for v in e.values())
sum_a=sum((v/stubs)**2 for v in a.values())
r=(sum_e-sum_a)/(1-sum_a)
print(r)

0.0018973173959750337


And the assortivity degree for this network is then 0.001897

> __Part 2: Configuration model__
> In the following, we are going to assess the significance of the assortativity by comparing the network's assortativity coefficient against that of random networks generated through the configuration model.  
>
> 2. *Implement the configuration model* using the _double edge swap_ algorithm to generate random networks. Ensure each node retains its original degree but with altered connections. Create a function that does that by following these steps:
>
>   - **a.** Create an exact copy of your original network.
>   - **b.** Select two edges, $e_{1} = (u,v)$ and $e_{2} = (x,y)$, ensuring *u != y* and *v != x*.
>   - **c.** Flip the direction of $e_{1}$ to $e_{1} = (v,u)$ 50% of the time. This ensure that your final results is not biased, in case your edges were sorted (they usually are). 
>   - **d.** Ensure that new edges $e_{1}' = (e_{1}[0],e_{2}[1])$ and $e_{2}' = (e_{2}[0],e_{1}[1])$ do not already exist in the network.
>   - **e.** Remove edges $e_{1}$ and $e_{2}$ and add edges $e_{1}'$ and $e_{2}'$.
>   - **f.** Repeat steps **b** to **e** until you have performed $E\cdot10$ swaps, where E is the total number of edges.
>

First we define a function to compute the edge swap

In [32]:
import random
def double_edge_swap(network,times=10):

    G_configuration = network.copy()
    edges = list(G_configuration.edges())  # build once

    num_edges = len(edges)

    for i in range(num_edges * times):

        # sample two edges uniformly
        (n1, n2) = random.choice(edges)
        (n3, n4) = random.choice(edges)

        # ensure the two edges are distinct
        if (n1, n2) == (n3, n4):
            continue

        # skip if they share a node
        if len({n1, n2, n3, n4}) < 4:
            continue

        # choose wiring pattern
        if random.random() < 0.5:
            a, b = n1, n3
            c, d = n2, n4
        else:
            a, b = n1, n4
            c, d = n2, n3

        # avoid self-loops and parallel edges
        if a == b or c == d:
            continue
        if G_configuration.has_edge(a, b) or G_configuration.has_edge(c, d):
            continue

        # perform swap
        G_configuration.remove_edge(n1, n2)
        G_configuration.remove_edge(n3, n4)
        G_configuration.add_edge(a, b)
        G_configuration.add_edge(c, d)

        # update the edges list incrementally
        edges.remove((n1, n2))
        edges.remove((n3, n4))
        edges.append((a, b))
        edges.append((c, d))
    return G_configuration

Then we can compute the configuration network

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

args = [G] * 5


with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(double_edge_swap, arg) for arg in args]

    configs = []
    for f in tqdm(as_completed(futures), total=len(futures)):
        configs.append(f.result())


  0%|          | 0/5 [00:00<?, ?it/s]

The output from this is from tqdm, which cannot be rendered when uploaded to learn. It is simply a progress bar, which tells us how far we are in the process, as it is very computationally expensive to compute this. 



> 3. *Double check that your algorithm works well*, by showing that the degree of nodes in the original network and the new 'randomized' version of the network are the same.
>
>
> __Part 3: Analyzing Assortativity in Random Networks__  
>
> 4. *Generate and analyze at least 100 random networks* using the configuration model. For each, calculate the assortativity with respect to the country and plot the distribution of these values. Compare the results with the assortativity of your original network to determine if connections within the same country are significantly higher than chance.
>
> __Part 4: Assortativity by Degree__
>
> 5. *Calculate degree assortativity* for your network using the formula discussed in the lecture.
> 6. *Compare your network's degree assortativity* against that of 100 random networks generated via the configuration model. Analyze whether your network shows a tendency for high-degree scientists to connect with other high-degree scientists and vice versa. 
>
> __Part 5: Reflection questions__    
> 7. *Assortativity by degree.* Were the results of the degree assortativity in line with your expectations? Why or why not?    
> 8. *Edge flipping.* In the process of implementing the configuration model, you were instructed to flip the edges (e.g., changing $e_1$ from (u,v) to (v,u)) 50% of the time. Why do you think this step is included?    
> 9. *Distribution of assortativity in random networks.* Describe the distribution of degree assortativity values you observed for the random 
networks. Was the distribution pattern expected? Discuss how the nature of random network generation (specifically, the configuration model and edge flipping) might influence this distribution and whether it aligns with theoretical expectations.    



# Part 2: TF-IDF

> __Exercise 1: TF-IDF and the Computational Social Science communities.__ The goal for this exercise is to find the words charachterizing each of the communities of Computational Social Scientists.
> What you need for this exercise: 
>.   
>    * The assignment of each author to their network community, and the degree of each author (Week 6, Exercise 4). This can be stored in a dataframe or in two dictionaries, as you prefer.  
>    * the tokenized _abstract_ dataframe (Week 7, Exercise 2)
>
> 1. First, check out [the wikipedia page for TF-IDF](https://en.wikipedia.org/wiki/Tf%E2%80%93idf). Explain in your own words the point of TF-IDF. 
>   * What does TF stand for? 
>   * What does IDF stand for?
> 2. Now, we want to find out which words are important for each *community*, so we're going to create several ***large documents, one for each community***. Each document includes all the tokens of abstracts written by members of a given community. 
>   * Consider a community _c_
>   * Find all the abstracts of papers written by a member of community _c_.
>   * Create a long array that stores all the abstract tokens 
>   * Repeat for all the communities. 
> __Note:__ Here, to ensure your code is efficient, you shall exploit ``pandas`` builtin functions, such as [``groupby.apply``](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.apply.html) or [``explode``](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.explode.html).
> 3. Now, we're ready to calculate the TF for each word. Use the method of your choice to find the top 5 terms within the __top 5 communities__ (by number of authors). 
>   * Describe similarities and differences between the communities.
>   * Why aren't the TFs not necessarily a good description of the communities?
>   * Next, we calculate IDF for every word. 
>   * What base logarithm did you use? Is that important?
> 4. We're ready to calculate TF-IDF. Do that for the __top 9 communities__ (by number of authors). Then for each community: 
>   * List the 10 top TF words 
>   * List the 10 top TF-IDF words
>   * List the top 3 authors (by degree)
>   * Are these 10 words more descriptive of the community? If yes, what is it about IDF that makes the words more informative?

 __Exercise 2: The Wordcloud__. It's time to visualize our results!

> * Install the [`WordCloud`](https://pypi.org/project/wordcloud/) module. 
> * Now, create word-cloud for each community. Feel free to make it as fancy or non-fancy as you like.
> * Make sure that, together with the word cloud, you print the names of the top three authors in each community (see my plot above for inspiration). 
> * Comment on your results. What can you conclude on the different sub-communities in Computational Social Science? 
> * Look up online the top author in each community. In light of your search, do your results make sense?

 __Exercise 3: Computational Social Science__ 

> * In light of your data-driven analysis, has your understanding of the field changed? How? 